# 스태커 크레인: S 자동 적재

빈 장면에서 만든 `stacker_crane_s_from_scratch.factoryio`와 함께 사용한다.
Factory I/O는 **Modbus TCP/IP Server**, 이 노트북은 클라이언트다.

1. 장면을 열고 Driver를 Start한다.
2. Edit → Run으로 **빈 장면**을 초기화한다.
3. 아래 설정과 초기 장면 확인 값을 바꾼다.
4. 실행 셀의 `IDLE` 출력 후 앞쪽 조작반의 **Start**를 누른다.

전체 설명과 주소표는 같은 폴더의 `README.md`에 있다. 이 노트북과 `controller.py`를 같은 폴더에 보관한다.

> **검증 상태:** 1번 칸 적재·99번 복귀 확인. Start 마우스 입력 및 S 전체 30개 안착은 미검증이다. `VALIDATION.md`를 먼저 확인한다. Page Up / Page Down으로 Overview / Controls 카메라를 전환할 수 있다.

이 사본은 기존 dtwin 커널의 PyModbus 3.16.2를 사용한다. 이번 실행 전 빈 장면과 정위치를 화면 및 입력으로 확인했다. 다음 실행 전에도 반드시 장면을 초기화한다.

## 1. 커널과 패키지
이 폴더의 `.venv/Scripts/python.exe` 커널을 선택하면 설치되어 있다. 다른 커널에서만 아래 설치 셀을 실행한다.

In [2]:
# 필요할 때만 주석 해제 후 실행
# %pip install pymodbus==3.11.1 ipykernel

In [3]:
from pathlib import Path
import sys
import pymodbus
print('Python:', sys.executable)
print('PyModbus:', pymodbus.__version__)
import inspect
from pymodbus.client import ModbusTcpClient
assert pymodbus.__version__.split('.')[0] == '3' and 'device_id' in inspect.signature(ModbusTcpClient.read_discrete_inputs).parameters, 'device_id를 지원하는 PyModbus 3.x가 필요합니다.'
assert Path('controller.py').exists(), '노트북의 작업 폴더를 controller.py가 있는 폴더로 바꾸세요.'
from controller import Settings, ModbusIO, StackerController, MASK, TARGETS

Python: D:\se4nchoi\intel26\digital_twin\1118\dtwin\Scripts\python.exe
PyModbus: 3.16.2


## 2. 접속 설정
Factory I/O Driver 화면의 서버 IP/Port/Slave ID와 맞춘다. API에는 `device_id`를 사용한다.

In [4]:
config = Settings(host='192.168.0.117', port=502, device_id=1)
EMPTY_SCENE_CONFIRMED = True  # 빈 장면을 직접 초기화한 뒤 True로 변경

## 3. 좌표 미리보기
우측 아래=1, 좌측 위=54. 6행 S는 30칸을 사용한다.

In [5]:
print('위 → 아래 / 왼쪽 → 오른쪽')
for z, row in zip(range(6, 0, -1), MASK):
    print(' '.join(f'{(z-1)*9+(10-x):02d}' if mark=='#' else '..' for x, mark in enumerate(row, 1)))
print('적재 순서:', list(TARGETS))
assert len(TARGETS) == len(set(TARGETS)) == 30

위 → 아래 / 왼쪽 → 오른쪽
54 53 52 51 50 49 48 47 46
45 .. .. .. .. .. .. .. ..
36 35 34 33 32 31 30 29 28
.. .. .. .. .. .. .. .. 19
.. .. .. .. .. .. .. .. 10
09 08 07 06 05 04 03 02 01
적재 순서: [46, 47, 48, 49, 50, 51, 52, 53, 54, 45, 36, 35, 34, 33, 32, 31, 30, 29, 28, 19, 10, 1, 2, 3, 4, 5, 6, 7, 8, 9]


## 4. 연결·입력 확인 (출력 변경 없음)
초기 입력은 보통 Sensor=False, MovingX/Z=False, Middle Limit=True다.
읽기 확인만으로 실제 원점과 빈 랙을 증명할 수는 없다.

In [6]:
probe = ModbusIO(config)
try:
    print(dict(zip(['Sensor','MovingX','MovingZ','Left Limit','Right Limit','Middle Limit','Start','Reset'], probe.inputs())))
finally:
    probe.close()

{'Sensor': False, 'MovingX': False, 'MovingZ': False, 'Left Limit': False, 'Right Limit': False, 'Middle Limit': True, 'Start': False, 'Reset': False}


## 5. 자동 적재 실행
실행 후 `IDLE`을 보고 Start 버튼을 누른다. 시작 버튼을 누르고 있으면 먼저 놓았다가 다시 누른다.
이 셀은 30개 적재가 끝날 때까지 실행 중 상태를 유지한다. 각 상태와 진행 개수가 출력된다.

**중단:** 장면의 Reset 또는 커널 Interrupt는 작업을 FAULT로 종료한다. 공급·이송은 끄고 크레인 출력은 보존한다.
현재 I/O에는 크레인 비상정지가 없으므로 이미 진행 중인 축 이동을 즉시 멈추려면 **Factory I/O Pause**를 누른다.
Target=0은 이전 이동을 취소하지 않는다. 재시작은 Edit → Run으로 빈 장면을 다시 만든 뒤 수행한다.

In [7]:
if not EMPTY_SCENE_CONFIRMED:
    raise RuntimeError('Factory I/O에서 빈 장면으로 초기화하고 EMPTY_SCENE_CONFIRMED=True로 바꾸세요.')
controller = StackerController(ModbusIO(config), config)
completed = controller.run(empty_scene_confirmed=True, wait_for_start=True)
print(f'완료: {len(completed)}/30, 로그: {controller.log}')

{"time": 1795653018.43523, "state": "IDLE", "completed": 0, "targets": [46, 47, 48, 49, 50, 51, 52, 53, 54, 45, 36, 35, 34, 33, 32, 31, 30, 29, 28, 19, 10, 1, 2, 3, 4, 5, 6, 7, 8, 9]}


## 6. 종료 결과
완료 좌표를 화면의 실제 팔레트 안착 상태와 함께 확인한다. 재실행하면 중복 적재되므로 반드시 빈 장면부터 초기화한다.

In [ ]:
if 'controller' in globals():
    print('상태:', controller.state)
    print('완료 좌표:', controller.completed)
    print('로그:', controller.log)